# ONNX Runtime 推理教程

本教程详细介绍 ONNX Runtime 推理引擎的使用，包括：

1. **ONNX 基础**: 模型格式和导出
2. **推理会话**: 创建和配置
3. **执行提供者**: CPU/GPU 加速
4. **性能优化**: 图优化和 IO Binding

---

## 什么是 ONNX Runtime？

ONNX Runtime 是微软开发的跨平台推理引擎：

```
PyTorch/TensorFlow → ONNX → ONNX Runtime → 多平台部署
                              │
                              ├─→ CPU (x86, ARM)
                              ├─→ GPU (CUDA, ROCm)
                              ├─→ NPU (各厂商)
                              └─→ Web (WebAssembly)
```

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import torch
import torch.nn as nn
import tempfile
import os
import time

# 检查 ONNX Runtime
try:
    import onnxruntime as ort
    print(f"ONNX Runtime 版本: {ort.__version__}")
    print(f"可用的执行提供者: {ort.get_available_providers()}")
except ImportError:
    print("请安装 ONNX Runtime: pip install onnxruntime")

## 1. 创建测试模型并导出为 ONNX

In [ ]:
# 定义测试模型
class ImageClassifier(nn.Module):
    """图像分类模型"""
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.pool = nn.MaxPool2d(2)
        self.fc1 = nn.Linear(64 * 8 * 8, 256)
        self.fc2 = nn.Linear(256, num_classes)
        self.dropout = nn.Dropout(0.5)
    
    def forward(self, x):
        x = self.pool(torch.relu(self.bn1(self.conv1(x))))
        x = self.pool(torch.relu(self.bn2(self.conv2(x))))
        x = x.view(x.size(0), -1)
        x = self.dropout(torch.relu(self.fc1(x)))
        return self.fc2(x)

# 创建模型
model = ImageClassifier()
model.eval()

# 统计参数
total_params = sum(p.numel() for p in model.parameters())
print(f"模型参数量: {total_params:,}")

In [ ]:
# 导出为 ONNX
dummy_input = torch.randn(1, 3, 32, 32)

# 创建临时目录保存模型
model_dir = tempfile.mkdtemp()
onnx_path = os.path.join(model_dir, "model.onnx")

torch.onnx.export(
    model,
    dummy_input,
    onnx_path,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={
        'input': {0: 'batch_size'},
        'output': {0: 'batch_size'}
    },
    opset_version=14
)

print(f"ONNX 模型已保存: {onnx_path}")
print(f"模型大小: {os.path.getsize(onnx_path) / (1024*1024):.2f} MB")

## 2. 创建推理会话

In [ ]:
# 基本推理会话
session = ort.InferenceSession(onnx_path)

# 查看输入输出信息
print("输入信息:")
for inp in session.get_inputs():
    print(f"  名称: {inp.name}")
    print(f"  形状: {inp.shape}")
    print(f"  类型: {inp.type}")

print("\n输出信息:")
for out in session.get_outputs():
    print(f"  名称: {out.name}")
    print(f"  形状: {out.shape}")
    print(f"  类型: {out.type}")

In [ ]:
# 执行推理
input_data = np.random.randn(1, 3, 32, 32).astype(np.float32)

# 方法 1: 使用字典
outputs = session.run(None, {'input': input_data})
print(f"输出形状: {outputs[0].shape}")
print(f"输出示例: {outputs[0][0][:5]}")

In [ ]:
# 验证与 PyTorch 输出一致
with torch.no_grad():
    torch_input = torch.from_numpy(input_data)
    torch_output = model(torch_input).numpy()

diff = np.abs(outputs[0] - torch_output).max()
print(f"PyTorch vs ONNX Runtime 最大差异: {diff:.6f}")
print(f"输出一致: {diff < 1e-5}")

## 3. 会话配置选项

In [ ]:
# 创建会话选项
sess_options = ort.SessionOptions()

# 图优化级别
# ORT_DISABLE_ALL: 禁用所有优化
# ORT_ENABLE_BASIC: 基本优化 (常量折叠等)
# ORT_ENABLE_EXTENDED: 扩展优化 (算子融合等)
# ORT_ENABLE_ALL: 所有优化
sess_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL

# 线程配置
sess_options.intra_op_num_threads = 4  # 算子内并行
sess_options.inter_op_num_threads = 2  # 算子间并行

# 执行模式
sess_options.execution_mode = ort.ExecutionMode.ORT_SEQUENTIAL

# 内存优化
sess_options.enable_mem_pattern = True
sess_options.enable_mem_reuse = True

# 创建优化后的会话
optimized_session = ort.InferenceSession(
    onnx_path,
    sess_options=sess_options,
    providers=['CPUExecutionProvider']
)

print("优化会话创建成功!")
print(f"使用的提供者: {optimized_session.get_providers()}")

In [ ]:
# 保存优化后的模型
optimized_path = os.path.join(model_dir, "model_optimized.onnx")

sess_options_save = ort.SessionOptions()
sess_options_save.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
sess_options_save.optimized_model_filepath = optimized_path

# 创建会话触发优化并保存
_ = ort.InferenceSession(onnx_path, sess_options=sess_options_save)

print(f"原始模型大小: {os.path.getsize(onnx_path) / 1024:.2f} KB")
print(f"优化模型大小: {os.path.getsize(optimized_path) / 1024:.2f} KB")

## 4. 执行提供者 (Execution Providers)

In [ ]:
# 查看可用的执行提供者
available_providers = ort.get_available_providers()
print("可用的执行提供者:")
for provider in available_providers:
    print(f"  - {provider}")

# 常见的执行提供者
provider_info = {
    'CPUExecutionProvider': 'CPU (默认)',
    'CUDAExecutionProvider': 'NVIDIA GPU',
    'TensorrtExecutionProvider': 'NVIDIA TensorRT',
    'ROCMExecutionProvider': 'AMD GPU',
    'DmlExecutionProvider': 'DirectML (Windows)',
    'CoreMLExecutionProvider': 'Apple CoreML',
    'OpenVINOExecutionProvider': 'Intel OpenVINO',
}

print("\n执行提供者说明:")
for provider, desc in provider_info.items():
    status = "✓" if provider in available_providers else "✗"
    print(f"  [{status}] {provider}: {desc}")

In [ ]:
# 使用特定的执行提供者
def create_session_with_provider(model_path, providers):
    """创建使用指定提供者的会话"""
    try:
        session = ort.InferenceSession(model_path, providers=providers)
        actual_providers = session.get_providers()
        print(f"请求: {providers}")
        print(f"实际: {actual_providers}")
        return session
    except Exception as e:
        print(f"创建失败: {e}")
        return None

# CPU 会话
print("=== CPU 会话 ===")
cpu_session = create_session_with_provider(onnx_path, ['CPUExecutionProvider'])

# 尝试 CUDA 会话 (如果可用)
if 'CUDAExecutionProvider' in available_providers:
    print("\n=== CUDA 会话 ===")
    cuda_session = create_session_with_provider(
        onnx_path, 
        ['CUDAExecutionProvider', 'CPUExecutionProvider']
    )

## 5. 性能基准测试

In [ ]:
def benchmark_session(session, input_data, num_runs=100, warmup=10):
    """基准测试推理性能"""
    input_name = session.get_inputs()[0].name
    
    # 预热
    for _ in range(warmup):
        session.run(None, {input_name: input_data})
    
    # 计时
    latencies = []
    for _ in range(num_runs):
        start = time.perf_counter()
        session.run(None, {input_name: input_data})
        latencies.append((time.perf_counter() - start) * 1000)
    
    return {
        'mean_ms': np.mean(latencies),
        'std_ms': np.std(latencies),
        'min_ms': np.min(latencies),
        'max_ms': np.max(latencies),
        'p50_ms': np.percentile(latencies, 50),
        'p90_ms': np.percentile(latencies, 90),
        'p99_ms': np.percentile(latencies, 99),
        'throughput': 1000 / np.mean(latencies)
    }

# 测试不同批次大小
batch_sizes = [1, 4, 8, 16, 32]
results = []

for batch_size in batch_sizes:
    input_data = np.random.randn(batch_size, 3, 32, 32).astype(np.float32)
    stats = benchmark_session(optimized_session, input_data, num_runs=50)
    stats['batch_size'] = batch_size
    stats['total_throughput'] = stats['throughput'] * batch_size
    results.append(stats)

# 显示结果
print(f"{'Batch':<8} {'Mean(ms)':<12} {'P50(ms)':<12} {'P99(ms)':<12} {'Throughput':<15}")
print("-" * 60)
for r in results:
    print(f"{r['batch_size']:<8} {r['mean_ms']:<12.3f} {r['p50_ms']:<12.3f} {r['p99_ms']:<12.3f} {r['total_throughput']:<15.1f}")

In [ ]:
# 可视化性能
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 延迟
axes[0].bar([str(r['batch_size']) for r in results], 
            [r['mean_ms'] for r in results], 
            color='steelblue', alpha=0.7)
axes[0].set_xlabel('Batch Size')
axes[0].set_ylabel('Latency (ms)')
axes[0].set_title('推理延迟 vs 批次大小')

# 吞吐量
axes[1].bar([str(r['batch_size']) for r in results], 
            [r['total_throughput'] for r in results], 
            color='coral', alpha=0.7)
axes[1].set_xlabel('Batch Size')
axes[1].set_ylabel('Throughput (samples/sec)')
axes[1].set_title('吞吐量 vs 批次大小')

plt.tight_layout()
plt.show()

## 6. 使用封装的推理模块

In [ ]:
from onnx_runtime import (
    ONNXInferenceSession,
    SessionConfig,
    Benchmarker,
    GraphOptimizationLevel,
    ExecutionMode,
    create_session,
    get_available_providers
)

# 使用封装的会话
config = SessionConfig(
    providers=['CPUExecutionProvider'],
    graph_optimization_level=GraphOptimizationLevel.ENABLE_ALL,
    intra_op_num_threads=4,
    execution_mode=ExecutionMode.SEQUENTIAL
)

inference_session = ONNXInferenceSession(onnx_path, config)

print(f"输入名称: {inference_session.input_names}")
print(f"输出名称: {inference_session.output_names}")
print(f"使用的提供者: {inference_session.get_providers()}")

In [ ]:
# 使用 Benchmarker
benchmarker = Benchmarker(inference_session)

# 测试不同批次大小
results = benchmarker.benchmark_batch_sizes(
    input_shape=(3, 32, 32),
    batch_sizes=[1, 4, 8, 16],
    num_runs=30
)

print("基准测试结果:")
for r in results:
    print(f"  Batch {r['batch_size']}: {r['mean_ms']:.2f}ms, {r['total_throughput']:.1f} samples/sec")

## 7. 动态输入形状

In [ ]:
# 测试动态批次大小
print("测试动态批次大小:")
for batch_size in [1, 5, 10, 20]:
    input_data = np.random.randn(batch_size, 3, 32, 32).astype(np.float32)
    output = inference_session.run_single(input_data)
    print(f"  输入: {input_data.shape} → 输出: {output.shape}")

## 8. 图优化级别对比

In [ ]:
# 测试不同优化级别
optimization_levels = [
    (ort.GraphOptimizationLevel.ORT_DISABLE_ALL, "禁用优化"),
    (ort.GraphOptimizationLevel.ORT_ENABLE_BASIC, "基本优化"),
    (ort.GraphOptimizationLevel.ORT_ENABLE_EXTENDED, "扩展优化"),
    (ort.GraphOptimizationLevel.ORT_ENABLE_ALL, "全部优化"),
]

input_data = np.random.randn(8, 3, 32, 32).astype(np.float32)
opt_results = []

for level, name in optimization_levels:
    opts = ort.SessionOptions()
    opts.graph_optimization_level = level
    
    sess = ort.InferenceSession(onnx_path, sess_options=opts)
    
    # 预热
    for _ in range(10):
        sess.run(None, {'input': input_data})
    
    # 计时
    times = []
    for _ in range(50):
        start = time.perf_counter()
        sess.run(None, {'input': input_data})
        times.append((time.perf_counter() - start) * 1000)
    
    opt_results.append({
        'name': name,
        'mean_ms': np.mean(times),
        'std_ms': np.std(times)
    })

# 显示结果
print(f"{'优化级别':<15} {'平均延迟(ms)':<15} {'标准差(ms)':<15}")
print("-" * 45)
for r in opt_results:
    print(f"{r['name']:<15} {r['mean_ms']:<15.3f} {r['std_ms']:<15.3f}")

## 总结

本教程介绍了 ONNX Runtime 的核心功能：

1. **模型导出**: PyTorch → ONNX
2. **推理会话**: 创建和配置
3. **执行提供者**: CPU/GPU 加速
4. **图优化**: 不同优化级别
5. **性能测试**: 基准测试方法

### 最佳实践

- 使用 `ORT_ENABLE_ALL` 优化级别
- 根据硬件选择合适的执行提供者
- 使用动态批次大小提高灵活性
- 通过基准测试找到最优配置

In [ ]:
# 清理临时文件
import shutil
shutil.rmtree(model_dir, ignore_errors=True)
print("临时文件已清理")